# Week 18, Day 6: Forge: Week 18 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt
print("1.", round(1 - 0.95 ** 20, 3))
print("2.", int(np.ceil(((1.959964 + 0.841621) / 0.4) ** 2)))
print("3.", 0.05 / 300)

In [ ]:
Phi = np.vectorize(lambda z: 0.5 * (1 + erf(z / sqrt(2))))
rng = np.random.default_rng(2013)
mice, per = 6, 12
mouse_effect = 0.3 + 1.0 * rng.normal(size=mice)            # mice differ a lot
change = mouse_effect[:, None] + 0.8 * rng.normal(size=(mice, per))
print("mouse averages:", change.mean(axis=1).round(2))

### Exercise: Part A: neurons, mice or both?

Compute `p_naive` (z-test on all 72 changes), `p_mouse` (sign-flip test on the 6 mouse averages, using all 64 sign patterns exactly), and 95% percentile intervals for the mean change from a naive bootstrap over neurons (`ci_naive`) and a hierarchical bootstrap over mice then neurons (`ci_hier`), each with 4,000 resamples.

In [ ]:
flat = change.ravel()
z = flat.mean() / (flat.std(ddof=1) / np.sqrt(flat.size))
p_naive = 2 * (1 - Phi(abs(z)))

m_avg = change.mean(axis=1)
signs = np.array([[1 if (k >> j) & 1 else -1 for j in range(mice)] for k in range(2 ** mice)])
p_mouse = ...                                   # fraction of the 64 sign patterns with |mean| at least the observed

boot_naive = flat[rng.integers(0, flat.size, (4000, flat.size))].mean(axis=1)
ci_naive = np.percentile(boot_naive, [2.5, 97.5])
boot_h = []
for _ in range(4000):
    pick = rng.integers(0, mice, mice)
    boot_h.append(...)                          # resample neurons within each picked mouse, then average
ci_hier = np.percentile(boot_h, [2.5, 97.5])
print(f"naive p = {p_naive:.4f};  mouse-level p = {p_mouse:.4f};  naive CI {ci_naive.round(2)};  hierarchical CI {ci_hier.round(2)}")

In [ ]:
import numpy as np
def test_naive():
    "the naive test treating 72 neurons as independent gives p below 0.05"
    assert p_naive < 0.05

def test_mouse():
    "the sign-flip test on the 6 mouse averages does not reach 0.05"
    assert p_mouse > 0.05

def test_hier():
    "the hierarchical bootstrap interval includes zero, and is more than twice as wide as the naive interval"
    assert ci_hier[0] < 0 < ci_hier[1] and np.diff(ci_hier)[0] > 2 * np.diff(ci_naive)[0]

test_naive()
test_mouse()
test_hier()
print("All checks passed")

In [ ]:
r_b = np.random.default_rng(0)
Xn = r_b.normal(size=(100, 200))                 # no information at all
yb = np.repeat([0, 1], 50)

def select(X, y, k=5):
    t = np.abs(X[y == 1].mean(0) - X[y == 0].mean(0)) / np.sqrt(X[y == 1].var(0) / (y == 1).sum() + X[y == 0].var(0) / (y == 0).sum())
    return np.argsort(t)[-k:]

def nearest_mean(Xtr, ytr, Xte):
    m0, m1 = Xtr[ytr == 0].mean(0), Xtr[ytr == 1].mean(0)
    return (((Xte - m1) ** 2).sum(1) < ((Xte - m0) ** 2).sum(1)).astype(int)

### Exercise: Part B: leaky and honest cross-validation

Write `cv_accuracy(X, y, honest, rng)` with 5 folds: if `honest`, select neurons on each training fold; otherwise select once on all data. Compute `acc_leaky` and `acc_honest`, then a permutation p-value `p_honest` for the honest pipeline from 100 shuffles of the labels.

In [ ]:
def cv_accuracy(X, y, honest, rng):
    folds = np.array_split(rng.permutation(len(y)), 5)
    feats_all = select(X, y)
    correct = 0
    for f in folds:
        tr = np.setdiff1d(np.arange(len(y)), f)
        feats = ...
        pred = nearest_mean(X[tr][:, feats], y[tr], X[f][:, feats])
        correct += np.sum(pred == y[f])
    return correct / len(y)

acc_leaky = cv_accuracy(Xn, yb, False, r_b)
acc_honest = cv_accuracy(Xn, yb, True, r_b)
null = [cv_accuracy(Xn, r_b.permutation(yb), True, r_b) for _ in range(100)]
p_honest = ...
print(f"leaky accuracy {acc_leaky:.2f};  honest accuracy {acc_honest:.2f} (permutation p = {p_honest:.2f})")

In [ ]:
import numpy as np
def test_leaky():
    "selecting neurons on all trials gives an accuracy well above chance (above 0.65) from pure noise"
    assert acc_leaky > 0.65

def test_honest():
    "selecting inside each fold gives an accuracy near chance (0.35 to 0.65)"
    assert 0.35 < acc_honest < 0.65

def test_perm():
    "the permutation test of the honest pipeline does not find the decoder significant"
    assert p_honest > 0.05

test_leaky()
test_honest()
test_perm()
print("All checks passed")

### Exercise: Part C: 300 permutation tests and BH

Simulate the 300 neurons (the first 60 tuned), compute each neuron's tuning statistic, and its permutation p-value from 500 shuffles of the orientation labels (use the same shuffles for every neuron, vectorised). Store `pvals`, then apply Benjamini-Hochberg at q = 0.05 to get `found`.

In [ ]:
r_c = np.random.default_rng(300)
n_neur, n_ori, n_tr = 300, 8, 10
tuned = np.arange(n_neur) < 60
ori = np.repeat(np.arange(n_ori), n_tr)                          # 80 trials
pref = r_c.integers(0, n_ori, n_neur)
mean_rate = 5 + 2.0 * tuned[:, None] * np.cos(2 * np.pi * (np.arange(n_ori)[None] - pref[:, None]) / n_ori)
R = r_c.poisson(mean_rate[:, ori])                                # neurons x trials

def tuning_stat(R, labels):
    means = np.stack([R[:, labels == o].mean(axis=1) for o in range(n_ori)], axis=1)
    return means.var(axis=1)

obs = tuning_stat(R, ori)
null = np.stack([tuning_stat(R, r_c.permutation(ori)) for _ in range(500)], axis=1)   # neurons x shuffles
pvals = ...

def benjamini_hochberg(p, q):
    m = len(p); order = np.argsort(p)
    passed = p[order] <= np.arange(1, m + 1) / m * q
    k = np.flatnonzero(passed).max() + 1 if passed.any() else 0
    out = np.zeros(m, bool); out[order[:k]] = True
    return out

found = ...
print(f"uncorrected: {np.sum(pvals < 0.05)} found, {np.sum((pvals < 0.05) & ~tuned)} false;  BH: {found.sum()} found, {np.sum(found & ~tuned)} false;  truly tuned: {tuned.sum()}")

In [ ]:
import numpy as np
def test_pvals():
    "there is one permutation p-value per neuron, all in (0, 1]"
    assert pvals.shape == (300,) and pvals.min() > 0 and pvals.max() <= 1

def test_bh_fdp():
    "BH discoveries are mostly real: false discovery proportion below 0.15"
    assert np.sum(found & ~tuned) / max(found.sum(), 1) < 0.15

def test_uncorrected_worse():
    "uncorrected p below 0.05 gives more false discoveries than BH"
    assert np.sum((pvals < 0.05) & ~tuned) > np.sum(found & ~tuned)

test_pvals()
test_bh_fdp()
test_uncorrected_worse()
print("All checks passed")